## Cleaned Tables

This notebook is part of a bigger team project where I am cleaning and combining multiple raw polling datasets from the 2016 U.S. Presidential Election. The goal is to merge all the usable tables into one clean, unified dataset with the goal of using it to build a baseline election prediction model.

## Why I’m Merging Everything Into One Dataset

Merging the cleaned datasets will help me:

- Track how support for candidates changed over time  
- Compare different polling methods and sources  
- Build a baseline model that uses the fullest version of the polling data  
- In the future, create new features like party averages, pollster bias, or time-to-election trends  

But this only works if the datasets line up cleanly. That means shared columns need to have the same names and data types. That’s why I’m cleaning each dataset first before trying to merge them.

In [1]:
import pandas as pd

---

## Cleaning Aggregate Polls in 2016 Data Set

In [2]:
df = pd.read_csv('../2016_raw_poll_tables/Aggregate_Polls_2016_raw.csv')
df.head()

,Race,Poll model,Hillary Clinton Democratic,Donald Trump Republican,Gary Johnson Libertarian,Jill Stein Green,Leading by (points)
0,Two-way,270toWin[4],47.2%,43.6%,—,—,3.6
1,Two-way,BBC[5],48.0%,44.0%,—,—,4.0
2,Two-way,HuffPost Pollster[6],47.3%,42.0%,—,—,5.3
3,Two-way,New York Times[7],45.9%,42.8%,—,—,3.1
4,Two-way,Real Clear Politics[8],46.8%,43.6%,—,—,3.2


In [3]:
# Check which columns need renaming 
df.columns

Index(['Race', 'Poll model', 'Hillary Clinton Democratic',
       'Donald Trump Republican', 'Gary Johnson Libertarian',
       'Jill Stein Green', 'Leading by (points)'],
      dtype='object')

In [4]:
# Create Dict to rename multiple columns at the same time
rename_dict = {
    'Hillary Clinton Democratic': 'Democrat',
    'Donald Trump Republican': 'Republican',
    'Gary Johnson Libertarian': 'Libertarian',
    'Jill Stein Green': 'Green',
    'Leading by (points)': 'Leading_Margin'
}

df.rename(columns=rename_dict, inplace=True)

In [5]:
df.dtypes

Race               object
Poll model         object
Democrat           object
Republican         object
Libertarian        object
Green              object
Leading_Margin    float64
dtype: object

In [7]:
# Convert columns to hold numeric values  
# Apply to the relevant columns
cols_to_convert = ['Democrat', 'Republican', 'Libertarian', 'Green']

# Create loop to convert values to strings and convert them to numeric later on, remove percent symbol, replace dashes with nothing 
for col in cols_to_convert:
    df[col] = df[col].astype(str).str.replace('%', '').replace('–', None)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [8]:
# Check
df.head()

,Race,Poll model,Democrat,Republican,Libertarian,Green,Leading_Margin
0,Two-way,270toWin[4],47.2,43.6,NaN,NaN,3.6
1,Two-way,BBC[5],48.0,44.0,NaN,NaN,4.0
2,Two-way,HuffPost Pollster[6],47.3,42.0,NaN,NaN,5.3
3,Two-way,New York Times[7],45.9,42.8,NaN,NaN,3.1
4,Two-way,Real Clear Politics[8],46.8,43.6,NaN,NaN,3.2


In [9]:
# Check (should all show float or int)
df[cols_to_convert].dtypes  # Nice

Democrat       float64
Republican     float64
Libertarian    float64
Green          float64
dtype: object

In [10]:
# Saved cleaned dataset to folder
df.to_csv('../2016_cleaned_poll_tables/Cleaned_Aggregate_Polls.csv', index=False)

### Dataset Overview

This dataset, Aggregate_Polls_2016_raw.csv, includes summary polling data from the 2016 election. It shows:

- Candidate names with their party (like Hillary Clinton Democratic)
- The percent of support each candidate got
- How much one candidate was leading by

### Why I Needed to Clean It

Each table uses full names like Donald Trump Republican instead of just saying the party. Since I'm going to merge this with other tables, I need the columns to be consistent across all of them.

So I renamed:
- Hillary Clinton to Democrat
- Donald Trump to Republican
- Gary Johnson to Libertarian
- Jill Stein to Green

I also cleaned up the values by:
- Removing percent signs
- Replacing blanks and dashes with NaN
- Converting everything to numbers so I can actually work with the data

Last thing, I renamed the Leading by (points) column to leading_margin to make it easier to merge later.

---

# Cleaning the Five Way Race in 2016 Data Set

In [25]:
df = pd.read_csv('../2016_raw_poll_tables/Five_way_race_2016_raw.csv')
df.head()

,Poll source,Date,Hillary Clinton Democratic,Donald Trump Republican,Gary Johnson Libertarian,Jill Stein Green,Evan McMullin Independent,Leading by (points),Sample size,Margin of error
0,Saint Leo University[578],"October 22–26, 2016",42%,31%,6%,1%,5%,11,1050,–
1,Public Policy Polling[149],"September 27–28, 2016",44%,40%,6%,1%,2%,4,933,±3.2%
2,Echelon Insights[151],"September 26–27, 2016",44%,39%,6%,2%,1%,5,1833,–
3,Public Policy Polling[214],"August 26–28, 2016",42%,37%,6%,4%,1%,5,881,±3.3%


In [26]:
rename_dict = {
    'Hillary Clinton Democratic': 'Democrat',
    'Donald Trump Republican': 'Republican',
    'Gary Johnson Libertarian': 'Libertarian',
    'Jill Stein Green': 'Green',
    'Evan McMullin Independent': 'Other',	
    'Leading by (points)': 'Leading_Margin'
}

df.rename(columns=rename_dict, inplace=True)

In [27]:
df.head()

,Poll source,Date,Democrat,Republican,Libertarian,Green,Other,Leading_Margin,Sample size,Margin of error
0,Saint Leo University[578],"October 22–26, 2016",42%,31%,6%,1%,5%,11,1050,–
1,Public Policy Polling[149],"September 27–28, 2016",44%,40%,6%,1%,2%,4,933,±3.2%
2,Echelon Insights[151],"September 26–27, 2016",44%,39%,6%,2%,1%,5,1833,–
3,Public Policy Polling[214],"August 26–28, 2016",42%,37%,6%,4%,1%,5,881,±3.3%


In [29]:
for col in ['Democrat', 'Republican', 'Libertarian', 'Green', 'Other', 'Leading_Margin']:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False).replace('–', None)
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [30]:
df.dtypes

Poll source        object
Date               object
Democrat            int64
Republican          int64
Libertarian         int64
Green               int64
Other               int64
Leading_Margin      int64
Sample size         int64
Margin of error    object
dtype: object